# Day 3-01｜Tracking 概念：IoU 與 ID 關聯

> Python「黑客松」實戰分析籃球運動數據  
> Notebook 以初學 Python 學員為主：先觀察、再改變少量變數、最後才串接。

目標：不用 ByteTrack 也先理解 tracking 的核心問題：第 t frame 的 box 要跟第 t+1 frame 的哪個 box 配對？

In [ ]:
from pathlib import Path
import sys

# 在 Colab 中先掛載 Drive；本機執行時會自動略過。
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機或 zip 解壓後執行時，從目前資料夾往上找。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

print("COURSE_ROOT =", COURSE_ROOT)
print("assets =", COURSE_ROOT / "assets")

In [ ]:
import numpy as np
import pandas as pd
from src.cv_utils import load_json


def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0


track_data = load_json(
    COURSE_ROOT / "assets" / "samples" / "sample_tracking_boxes.json"
)
frame0 = track_data["frames"][0]["detections"]
frame1 = track_data["frames"][1]["detections"]

matrix = np.zeros((len(frame0), len(frame1)))
for i, a in enumerate(frame0):
    for j, b in enumerate(frame1):
        matrix[i, j] = iou(a["bbox_xyxy"], b["bbox_xyxy"])

pd.DataFrame(
    matrix.round(3),
    index=[f"f0_box{i}" for i in range(len(frame0))],
    columns=[f"f1_box{j}" for j in range(len(frame1))],
)

In [ ]:
# 最簡單的 greedy 配對：每個 f0 box 找 IoU 最大的 f1 box。
assignments = []
for i in range(matrix.shape[0]):
    j = int(matrix[i].argmax())
    assignments.append(
        {"track_id": i, "frame0_box": i, "frame1_box": j, "iou": matrix[i, j]}
    )

pd.DataFrame(assignments)

ByteTrack 比這個更完整，會處理低分數 detection、暫時消失、重複框等問題。但初學者先理解 IoU 關聯就夠了。